# Four-model ink ensemble (fast: seed 42 instead of v8-in) on `tifxyz/quick_patches_from_D`

Models: dense_native, Hecate, ink_9um seed 42, ink_9um seed 43. The 22 PHerc0846A quick patches are already assembled (`ves_zarrs2/<id>.zarr`, `masks/extras/<id>.png`).
Each segment saves `<id>_4models_d+0.png` and `<id>_ensemble_d+0.png` (ensemble over the composite view) to `output/test_visualizations/villa_ensemble/quick_patches/`. Finished segments are skipped; failures are retried once at the end.

In [1]:
import gc
import importlib
import os
import subprocess
import sys
import time

if sys.executable.startswith("/opt/villa-venv"):
    raise RuntimeError("select the system Python 3.10 kernel (/usr/bin/python3)")

REPO = "/vesuvius"
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import assemble_test_segments as ats_mod
from utils import villa_ensemble as ve
ve = importlib.reload(ve)

SLOT3 = "ink9-42"
ve.set_slot3(SLOT3)
ZARR_DIR = os.getenv("VESUVIUS_ZARR_PATH", os.path.join(REPO, "ves_zarrs2"))
OUT_DIR = os.path.join(REPO, "output", "test_visualizations", "villa_ensemble", "quick_patches")
DEPTHS = (0,)
WRITE_DEPTH = 0
DIRECTION = "forward"
COMPILE = False
TAG = f"d{WRITE_DEPTH:+d}"
TITLES = {"dnative": "dense_native (16k)", "hecate": "Hecate 9.6µm", "ink9-42": "ink_9um seed 42",
          "v8in-1447": "v8-in PHerc1447 LOO-w062", "ink9-43": "ink_9um seed 43"}
PATCHES = [(zid, sub.split(os.sep)[1]) for zid, sub, *_rest, set_name in ats_mod.extra_fragments()
           if set_name == "quick_patches_from_D"]


def outputs(zid):
    return [os.path.join(OUT_DIR, f"{zid}_{kind}_{TAG}.png") for kind in ("4models", "ensemble")]


def render(zid, scroll):
    t0 = time.time()
    ve.ensure_villa_input(zid, ZARR_DIR)
    ve.run_models(zid, DEPTHS, [DIRECTION], compile_model=COMPILE)
    stretched, ensemble, stats = ve.ensemble(zid, DEPTHS, DIRECTION, WRITE_DEPTH)
    grid_path, ens_path = outputs(zid)
    ve.save_grid([(f"{scroll} {zid}  {TITLES[k]}  {TAG} {DIRECTION}", stretched[k]) for k, _ in ve.MODELS], 2,
                 grid_path)
    ve.save_ensemble_figure(zid, ensemble, f"{scroll} {zid}  4-model ensemble (mean z-score)  {TAG} {DIRECTION}",
                            ens_path)
    del stretched, ensemble, stats
    gc.collect()
    return time.time() - t0


print(f"models: {[short for short, _ in ve.MODELS]} | {len(PATCHES)} quick patches | out: {OUT_DIR}")

models: ['dnative', 'hecate', 'ink9-42', 'ink9-43'] | 22 quick patches | out: /vesuvius/output/test_visualizations/villa_ensemble/quick_patches


In [2]:
from tqdm.auto import tqdm

RESULTS = {}
for attempt in (1, 2):
    todo = [(zid, scroll) for zid, scroll in PATCHES if not all(os.path.isfile(p) for p in outputs(zid))]
    for zid, scroll in tqdm(todo, desc=f"quick patches (pass {attempt})", unit="segment"):
        try:
            RESULTS[zid] = f"OK {render(zid, scroll):.0f}s"
        except Exception as exc:
            RESULTS[zid] = f"FAILED: {exc}"
            print(f"[FAILED] {zid}: {exc}", flush=True)
        print(f"[{zid}] {RESULTS[zid]}", flush=True)
failed = {z: r for z, r in RESULTS.items() if r.startswith("FAILED")}
print(f"done: {sum(all(os.path.isfile(p) for p in outputs(z)) for z, _ in PATCHES)}/{len(PATCHES)} | failed: {failed}")

quick patches (pass 1):   0%|          | 0/13 [00:00<?, ?segment/s]


auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:00<?] 
auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:11<?] , hybrid_3d2d-seed43__L28s6.tif 0/5600 patches
auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:14<5:47:58] , hybrid_3d2d-seed43__L28s6.tif 4/5600 patches
auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:14<2:54:04] , hybrid_3d2d-seed43__L28s6.tif 8/5600 patches
auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:14<1:56:06] , hybrid_3d2d-seed43__L28s6.tif 12/5600 patches
auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:14<1:27:08] , hybrid_3d2d-seed43__L28s6.tif 16/5600 patches
auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:15<1:09:47] , hybrid_3d2d-seed43__L28s6.tif 20/5600 patches
auto_grown_20260926155339746_abf:   0%|          | 0.00/1 maps [00:15<58:13] , hybrid_3d2d-seed43__L28s6.tif 24/5600 patches  
auto_grown_20260926155339746_abf:   0%|      

[villa] auto_grown_20260926155339746_abf: 1 maps in 50s -> /vesuvius/output/villa_ensemble_maps/auto_grown_20260926155339746_abf
[auto_grown_20260926155339746_abf] OK 105s



auto_grown_20260927093812925_abf:   0%|          | 0.00/4 maps [00:00<?] 
auto_grown_20260927093812925_abf:   0%|          | 0.00/4 maps [00:07<?] , dnative__L28s6.tif 0/1967 patches
auto_grown_20260927093812925_abf:   0%|          | 0.00/4 maps [00:10<5:33:41] , dnative__L28s6.tif 4/1967 patches
auto_grown_20260927093812925_abf:   0%|          | 0.00/4 maps [00:10<2:47:01] , dnative__L28s6.tif 8/1967 patches
auto_grown_20260927093812925_abf:   0%|          | 0.01/4 maps [00:10<1:51:28] , dnative__L28s6.tif 12/1967 patches
auto_grown_20260927093812925_abf:   0%|          | 0.01/4 maps [00:10<1:23:47] , dnative__L28s6.tif 16/1967 patches
auto_grown_20260927093812925_abf:   0%|          | 0.01/4 maps [00:10<1:07:11] , dnative__L28s6.tif 20/1967 patches
auto_grown_20260927093812925_abf:   0%|          | 0.01/4 maps [00:10<56:05] , dnative__L28s6.tif 24/1967 patches  
auto_grown_20260927093812925_abf:   0%|          | 0.01/4 maps [00:10<48:11] , dnative__L28s6.tif 28/1967 patches
auto_gro

[villa] auto_grown_20260927093812925_abf: 4 maps in 136s -> /vesuvius/output/villa_ensemble_maps/auto_grown_20260927093812925_abf
[auto_grown_20260927093812925_abf] OK 159s


...

[villa] auto_grown_20260927210402707_abf: 4 maps in 159s -> /vesuvius/output/villa_ensemble_maps/auto_grown_20260927210402707_abf
[auto_grown_20260927210402707_abf] OK 189s



auto_grown_20260927214603141_abf:   0%|          | 0.00/4 maps [00:00<?] 
auto_grown_20260927214603141_abf:   0%|          | 0.00/4 maps [00:06<?] , dnative__L28s6.tif 0/1431 patches
auto_grown_20260927214603141_abf:   0%|          | 0.00/4 maps [00:09<3:37:15] , dnative__L28s6.tif 4/1431 patches
auto_grown_20260927214603141_abf:   0%|          | 0.01/4 maps [00:09<1:48:45] , dnative__L28s6.tif 8/1431 patches
auto_grown_20260927214603141_abf:   0%|          | 0.01/4 maps [00:09<1:12:36] , dnative__L28s6.tif 12/1431 patches
auto_grown_20260927214603141_abf:   0%|          | 0.01/4 maps [00:09<54:31] , dnative__L28s6.tif 16/1431 patches  
auto_grown_20260927214603141_abf:   0%|          | 0.01/4 maps [00:09<43:43] , dnative__L28s6.tif 20/1431 patches
auto_grown_20260927214603141_abf:   0%|          | 0.02/4 maps [00:09<36:30] , dnative__L28s6.tif 24/1431 patches
auto_grown_20260927214603141_abf:   0%|          | 0.02/4 maps [00:09<31:22] , dnative__L28s6.tif 28/1431 patches
auto_grown_2

[villa] auto_grown_20260927214603141_abf: 4 maps in 102s -> /vesuvius/output/villa_ensemble_maps/auto_grown_20260927214603141_abf
[auto_grown_20260927214603141_abf] OK 115s


quick patches (pass 2): 0segment [00:00, ?segment/s]

done: 22/22 | failed: {}
